## Parameters

In [0]:
dbutils.widgets.text("start_date", "2026-09-01")
dbutils.widgets.text("end_date", "2026-09-02")

start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

print("Start date:", start_date)
print("End date:", end_date)

Start date: 2026-09-01
End date: 2026-09-02


instead of entering data manually , later we will focus on that architecture : 

Azure Data Factory
        ->
start_date / end_date
        ->
Databricks notebook

So we make the notebook parameterized from the beginning

# Bronze — call the USGS API

In [0]:
import requests

USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",
    "starttime": start_date,
    "endtime": end_date,
    "eventtype": "earthquake"
}

response = requests.get(
    USGS_URL,
    params=params,
    timeout=60
)

response.raise_for_status()

data = response.json()

print("HTTP status:", response.status_code)
print("Number of earthquakes:", len(data.get("features", [])))

HTTP status: 200
Number of earthquakes: 362


Examine the API response

In [0]:
data.keys()

dict_keys(['type', 'metadata', 'features', 'bbox'])

# Bronze — store the RAW GeoJSON

We are therefore going to store the complete GeoJSON response instead of flattening it immediately

In [0]:
import os
import json

CATALOG = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

SCHEMA = "earthquake"

BRONZE_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/bronze"

BRONZE_RUN_DIR = (
    f"{BRONZE_ROOT}/run_date={start_date}"
)

os.makedirs(
    BRONZE_RUN_DIR,
    exist_ok=True
)

BRONZE_FILE = (
    f"{BRONZE_RUN_DIR}/earthquakes.geojson"
)

with open(
    BRONZE_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        data,
        f,
        ensure_ascii=False
    )

print("Saved to:")
print(BRONZE_FILE)

Saved to:
/Volumes/azure_e2e_project/earthquake/bronze/run_date=2026-09-01/earthquakes.geojson


verify ;

In [0]:
os.listdir(BRONZE_RUN_DIR)

['earthquakes.geojson']

# Read Bronze file with Spark

In [0]:
bronze_df = (
    spark.read
    .option("multiline", "true")
    .json(BRONZE_FILE)
)

display(bronze_df)

bbox features metadata type List(-179.6286, -29.4696, -1.82, 169.7122, 67.413, 533.035) List(List(List(List(-116.808666666667, 33.7041666666667, 15.95), Point), ci41540448, List(null, null, 41540448, https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci41540448&format=geojson, 0.07908, null, 149, ,ci41540448,, 0.52, ml, null, ci, 15, 9 km WSW of Idyllwild, CA, 0.08, 4, ,ci,, reviewed, 1788307126050, M 0.5 - 9 km WSW of Idyllwild, CA, 0, earthquake, ,nearby-cities,origin,phase-data,, null, 1788549183582, https://earthquake.usgs.gov/earthquakes/eventpage/ci41540448), Feature), List(List(List(-98.094, 28.895, 10.7377), Point), tx2026rfxdkd, List(null, null, 2026rfxdkd, https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=tx2026rfxdkd&format=geojson, 0.1, null, 70, ,tx2026rfxdkd,, 1.7, ml, null, tx, 15, 12 km SW of Falls City, Texas, 0.2, 44, ,tx,, reviewed, 1788307106193, M 1.7 - 12 km SW of Falls City, Texas, 0, earthquake, ,origin,phase-data,, null, 1788360642505, https://earthquake.usgs.gov/earthquakes/eventpage/tx2026rfxdkd), Feature), List(List(List(-116.756166666667, 33.5071666666667, 10.2), Point), ci41540440, List(null, null, 41540440, https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci41540440&format=geojson, 0.04381, null, 31, ,ci41540440,, 1.17, ml, null, ci, 51, 9 km SW of Anza, CA, 0.18, 21, ,ci,, reviewed, 1788307064440, M 1.2 - 9 km SW of Anza, CA, 0, earthquake, ,focal-mechanism,nearby-cities,origin,phase-data,, null, 1788308880260, https://earthquake.usgs.gov/earthquakes/eventpage/ci41540440), Feature), List(List(List(-98.097, 28.894, 8.3691), Point), tx2026rfwxvb, List(null, null, 2026rfwxvb, https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=tx2026rfwxvb&format=geojson, 0.1, null, 70, ,tx2026rfwxvb,, 1.5, ml, null, tx, 19, 12 km SW of Falls City, Texas, 0.2, 35, ,tx,, reviewed, 1788306719633, M 1.5 - 12 km SW of Falls City, Texas, 0, earthquake, ,origin,phase-data,, null, 1788381358914, https://earthquake.usgs.gov/earthquakes/eventpage/tx2026rfwxvb), Feature), List(List(List(-149.619, 62.399, 56.4), Point), aka2026riqznl, List(null, null, a2026riqznl, https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=aka2026riqznl&format=geojson, 0.3, null, 44, ,aka2026riqznl,, 1.7, ml, null, ak, 36, 25 km ESE of Chase, Alaska, 0.6, 44, ,ak,, reviewed, 1788306567495, M 1.7 - 25 km ESE of Chase, Alaska, 0, earthquake, ,origin,phase-data,, null, 1788373198767, https://earthquake.usgs.gov/earthquakes/eventpage/aka2026riqznl), Feature), List(List(List(-117.471666666667, 34.2275, 8.28), Point), ci41540432, List(null, null, 41540432, https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci41540432&format=geojson, 0.03046, null, 36, ,ci41540432,, 0.76, ml, null, ci, 30, 4 km SE of Lytle Creek, CA, 0.15, 9, ,ci,, reviewed, 1788306554540, M 0.8 - 4 km SE of Lytle Creek, CA, 0, earthquake, ,nearby-cities,origin,phase-data,, null, 1788354493553, https://earthquake.usgs.gov/earthquakes/eventpage/ci41540432), Feature), List(List(List(-179.6286, -25.2424, 525.844), Point), us7000tfwm, List(null, null, 7000tfwm, https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us7000tfwm&format=geojson, 7.766, null, 73, ,us7000tfwm,, 4.5, mb, null, us, 30, south of the Fiji Islands, 0.63, 312, ,us,, reviewed, 1788306391422, M 4.5 - south of the Fiji Islands, 0, earthquake, ,origin,phase-data,, null, 1789076571040, https://earthquake.usgs.gov/earthquakes/eventpage/us7000tfwm), Feature), List(List(List(66.6421, -13.1772, 10.0), Point), us7000tdhg, List(null, null, 7000tdhg, https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us7000tdhg&format=geojson, 7.254, null, 57, ,us7000tdhg,, 4.6, mb, null, us, 43, Mid-Indian Ridge, 0.43, 326, ,us,, reviewed, 1788306323273, M 4.6 - Mid-Indian Ridge, 0, earthquake, ,origin,phase-data,, null, 1789075901040, https://earthquake.usgs.gov/earthquakes/eventpage/us7000tdhg), Feature), List(List(List(-155.307, 58.181, 2.2), Point), aka2026riqsyf, List(null, null, a2026riqsyf, https://

In [0]:
bronze_df.printSchema()

root
 |-- bbox: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- features: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- geometry: struct (nullable = true)
 |    |    |    |-- coordinates: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- type: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- properties: struct (nullable = true)
 |    |    |    |-- alert: string (nullable = true)
 |    |    |    |-- cdi: double (nullable = true)
 |    |    |    |-- code: string (nullable = true)
 |    |    |    |-- detail: string (nullable = true)
 |    |    |    |-- dmin: double (nullable = true)
 |    |    |    |-- felt: long (nullable = true)
 |    |    |    |-- gap: long (nullable = true)
 |    |    |    |-- ids: string (nullable = true)
 |    |    |    |-- mag: double (nullable = true)
 |    |    |    |-- magType: string (nullable = true)


Bronze layer finished ; yayy

Check Azure Storage
![image_1789136387887.png](./image_1789136387887.png "image_1789136387887.png")

Databricks wrote data
        ->
Unity Catalog Volume
        ->
External Location
        ->
ADLS Gen2